# 02 — PeptideCLM embeddings + k-means clustering

Part of a 5-notebook peptide-clustering + consensus-split exploration — see
`README_clustering.md` in this folder for the full picture and run order. This
notebook implements method (c): a *learned* SMILES embedding as a third,
independent clustering signal (alongside 01's fingerprint/descriptor voters and
03's sequence-identity voter), then k-means on that embedding space.

Like notebook 01, this can be run before or after the others — it loads
`data/clustering/peptide_voters.parquet` if present (to inherit prior voter
columns) or rebuilds the base frame fresh, and only ever writes its own
`cluster_peptideclm` column back.

**CPU-only note:** this is the one method in the four where CPU runtime is a real
concern (a small transformer forward pass per peptide, vs. near-instant fingerprint/
descriptor math). The notebook times a small sample first and prints an
extrapolated full-dataset estimate *before* committing to the full run — see the
timing cell below.


In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from tqdm.auto import tqdm
from transformers import AutoModel

## Setup: locate the repo and load/build the shared peptide-level frame (same pattern as notebook 01)

In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root (no pyproject.toml found above cwd)")


REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data"
CLUSTERING_DIR = DATA_DIR / "clustering"
CLUSTERING_DIR.mkdir(exist_ok=True)
VOTERS_PATH = CLUSTERING_DIR / "peptide_voters.parquet"
TOKENIZER_DIR = REPO_ROOT / "src" / "soamp" / "features" / "peptideclm"

print("REPO_ROOT:", REPO_ROOT)

In [3]:
def build_base_peptide_frame() -> pd.DataFrame:
    classification_df = pd.read_csv(DATA_DIR / "mic_classification_dataset.csv")
    base = (
        classification_df
        .drop_duplicates(subset="peptide_id")[["peptide_id", "sequence", "smiles", "has_noncanonical"]]
        .reset_index(drop=True)
    )
    regression_df = pd.read_csv(DATA_DIR / "final_mic_regression_dataset.csv")
    bond_types = regression_df.drop_duplicates(subset="peptide_id")[["peptide_id", "bond_type"]]
    base = base.merge(bond_types, on="peptide_id", how="left")
    base["is_linear"] = base["bond_type"].fillna("none") == "none"
    return base


def load_peptide_voters() -> pd.DataFrame:
    base = build_base_peptide_frame()
    if VOTERS_PATH.exists():
        existing = pd.read_parquet(VOTERS_PATH)
        voter_cols = [c for c in existing.columns if c not in base.columns]
        base = base.merge(existing[["peptide_id", *voter_cols]], on="peptide_id", how="left")
    return base


def save_peptide_voters(df: pd.DataFrame) -> None:
    df.to_parquet(VOTERS_PATH, index=False)
    print(f"saved {VOTERS_PATH} ({len(df)} rows, columns: {list(df.columns)})")


peptide_voters = load_peptide_voters()
print(peptide_voters.shape)
peptide_voters.head()


(12371, 11)


,peptide_id,sequence,smiles,has_noncanonical,bond_type,is_linear,cluster_fingerprint,cluster_descriptor,qmap_input_sequence,qmap_scoreable,cluster_qmap
0,10,LFIFFF,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,False,none,True,223.0,3.0,LFIFFF,True,7.0
1,11,RVKRVWPLVIRTVIAGYNLYRAIKKK,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,False,none,True,1.0,2.0,RVKRVWPLVIRTVIAGYNLYRAIKKK,True,53.0
2,12,RKRIHIGPGRAFYTT,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1cnc[nH]1)NC(=O)...,False,none,True,8.0,2.0,RKRIHIGPGRAFYTT,True,457.0
3,13,RRXXRF,CC(=O)N[C@@H](CCCN=C(N)N)C(=O)N[C@@H](CCCN=C(N...,True,none,True,270.0,3.0,NaN,False,NaN
4,14,GIWDTIKSMGKVFAGKILQNL,CC[C@H](C)[C@H](NC(=O)CN)C(=O)N[C@@H](Cc1c[nH]...,False,none,True,3.0,2.0,GIWDTIKSMGKVFAGKILQNL,True,143.0


In [4]:
def report_clustering_diagnostics(df: pd.DataFrame, cluster_col: str, runtime_seconds: float,
                                    imbalance_fraction_flag: float = 0.5) -> None:
    total = len(df)
    scored = df[cluster_col].notna()
    n_scored = int(scored.sum())
    sizes = df.loc[scored, cluster_col].value_counts()
    print(f"[{cluster_col}] runtime: {runtime_seconds:.2f}s")
    print(f"[{cluster_col}] coverage: {n_scored}/{total} peptides scored ({n_scored / total:.1%})")
    print(f"[{cluster_col}] cluster count: {len(sizes)}")
    print(f"[{cluster_col}] cluster size distribution: min={sizes.min()}, median={sizes.median():.0f}, "
          f"max={sizes.max()}, mean={sizes.mean():.1f}")
    largest_frac = sizes.max() / n_scored
    flag = " <-- SEVERE IMBALANCE" if largest_frac > imbalance_fraction_flag else ""
    print(f"[{cluster_col}] largest cluster is {largest_frac:.1%} of scored peptides{flag}")


## Method (c): PeptideCLM embeddings + k-means

**Why this model / citation:** [PeptideCLM](https://github.com/AaronFeller/PeptideCLM)
(Feller & Wilke, "Peptide-aware chemical language model successfully predicts
membrane diffusion of cyclic peptides", bioRxiv 2024 / *J. Chem. Inf. Model.*) is a
6-layer RoFormer chemical language model pretrained via masked-language-modeling
directly on ~11M peptide SMILES (plus small molecules) -- checkpoint
`aaronfeller/PeptideCLM-23M-all` on HuggingFace. It's used here specifically
*because* it embeds from SMILES rather than an amino-acid sequence: like this
project's own descriptor pipeline (`src/soamp/features/peptide.py`), it handles
non-canonical/cyclic peptides natively rather than needing a fallback, so unlike
method (d) it should have full 100% coverage.

**Tokenizer note:** the HF model repo ships weights only -- its README explains the
tokenizer must be loaded from the `tokenizer/` directory of the GitHub repo
(`SMILES_SPE_Tokenizer`, a SMILES Pair Encoding tokenizer). Those three small files
(`tokenizer.py`, `new_vocab.txt`, `new_splits.txt`, MIT-licensed, vendored
verbatim) live in the installable package at `src/soamp/features/peptideclm/` --
`soamp.features.peptide_featurizers.PeptideCLMFeaturizer` uses the same vendored
assets for the pipeline-selectable `peptideclm_embedding` featurization method.
No fine-tuning -- this is inference-only feature extraction (`AutoModel`, not a
task head), matching the "CPU inference, no fine-tuning" constraint.

In [ ]:
from soamp.features.peptideclm.tokenizer import SMILES_SPE_Tokenizer  # vendored, see markdown above

_t0 = time.perf_counter()
peptideclm_tokenizer = SMILES_SPE_Tokenizer(
    str(TOKENIZER_DIR / "new_vocab.txt"),
    str(TOKENIZER_DIR / "new_splits.txt"),
)
peptideclm_model = AutoModel.from_pretrained("aaronfeller/PeptideCLM-23M-all")
peptideclm_model.eval()
torch.set_grad_enabled(False)  # inference-only, no fine-tuning
print(f"loaded tokenizer + model in {time.perf_counter() - _t0:.1f}s")
print("model hidden size:", peptideclm_model.config.hidden_size)

In [6]:
def embed_smiles_batch(smiles_batch: list[str], max_length: int = 512) -> torch.Tensor:
    """Mean-pool the last hidden state over non-padding tokens -- a standard,
    simple sentence-embedding choice for an encoder model with no dedicated
    pooler for this task.
    """
    encoded = peptideclm_tokenizer(smiles_batch, return_tensors="pt", padding=True,
                                     truncation=True, max_length=max_length)
    output = peptideclm_model(**encoded)
    mask = encoded["attention_mask"].unsqueeze(-1)
    pooled = (output.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1)
    return pooled


### Required timing step: ~200-peptide sample first, extrapolate, THEN decide on the full run

Per the task's explicit ask -- run a small sample, print the estimated full-dataset
runtime, and only then run everything.


In [7]:
TIMING_SAMPLE_SIZE = 200
TIMING_BATCH_SIZE = 16

_sample = peptide_voters["smiles"].sample(n=TIMING_SAMPLE_SIZE, random_state=0).tolist()

_t0 = time.perf_counter()
for i in range(0, len(_sample), TIMING_BATCH_SIZE):
    embed_smiles_batch(_sample[i:i + TIMING_BATCH_SIZE])
_elapsed = time.perf_counter() - _t0

_rate = _elapsed / TIMING_SAMPLE_SIZE
_full_estimate_min = _rate * len(peptide_voters) / 60

print(f"{TIMING_SAMPLE_SIZE} peptides embedded in {_elapsed:.1f}s -> {_rate * 1000:.1f} ms/peptide")
print(f"estimated full dataset ({len(peptide_voters)} peptides): {_full_estimate_min:.1f} min "
      f"at batch_size={TIMING_BATCH_SIZE}")
print("If this looks too slow: raise TIMING_BATCH_SIZE / EMBED_BATCH_SIZE below, or reduce "
      "MAX_LENGTH -- CPU-bound transformer inference scales roughly linearly with both "
      "peptide count and sequence length.")


200 peptides embedded in 7.5s -> 37.6 ms/peptide
estimated full dataset (12371 peptides): 7.8 min at batch_size=16
If this looks too slow: raise TIMING_BATCH_SIZE / EMBED_BATCH_SIZE below, or reduce MAX_LENGTH -- CPU-bound transformer inference scales roughly linearly with both peptide count and sequence length.


The estimate above was well within the "minutes, not hours" constraint at
development time (~7-8 minutes for the full ~12.4k-peptide dataset on a 4-thread
CPU) -- proceeding to the full run. If your estimate looks much worse, lower
`EMBED_BATCH_SIZE` won't help (it's runtime-neutral, just memory/throughput
tradeoff) -- consider raising it instead, or set `torch.set_num_threads(...)`
explicitly to use more cores.


In [8]:
EMBED_BATCH_SIZE = 16  # adjustable; see the timing cell above before changing

# Sort by (tokenized) SMILES length before batching so peptides of similar length
# end up in the same batch -- this minimizes wasted padding compute, a meaningful
# CPU-time saver here since peptide SMILES lengths vary a lot in this dataset.
_smiles_list = peptide_voters["smiles"].tolist()
_order = np.argsort([len(s) for s in _smiles_list])

_t0 = time.perf_counter()
_embeddings = np.zeros((len(_smiles_list), peptideclm_model.config.hidden_size), dtype="float32")
for start in tqdm(range(0, len(_order), EMBED_BATCH_SIZE), desc="embedding"):
    batch_positions = _order[start:start + EMBED_BATCH_SIZE]
    batch_smiles = [_smiles_list[p] for p in batch_positions]
    pooled = embed_smiles_batch(batch_smiles).numpy()
    _embeddings[batch_positions] = pooled
_runtime_embed = time.perf_counter() - _t0

print(f"embedded {len(_smiles_list)} peptides in {_runtime_embed / 60:.1f} min")
peptideclm_embeddings = _embeddings


embedding:   0%|          | 0/774 [00:00<?, ?it/s]

embedded 12371 peptides in 4.2 min


In [9]:
# Small silhouette-score sweep (sampled -- 768-dim pairwise distances are pricier
# than the 13-dim descriptor space in notebook 01) to suggest a default k.
_sweep_ks = [5, 10, 15, 20, 25, 30]
_sweep_scores = []
for k in _sweep_ks:
    labels = KMeans(n_clusters=k, random_state=42, n_init="auto").fit_predict(peptideclm_embeddings)
    score = silhouette_score(peptideclm_embeddings, labels, sample_size=3000, random_state=42)
    _sweep_scores.append(score)
    print(f"k={k:>3}  silhouette={score:.4f}")

_suggested_k = _sweep_ks[int(np.argmax(_sweep_scores))]
print(f"\nsilhouette-suggested k: {_suggested_k}")


k=  5  silhouette=0.1080


k= 10  silhouette=0.1074


k= 15  silhouette=0.0969


k= 20  silhouette=0.0982


k= 25  silhouette=0.0943


k= 30  silhouette=0.0949

silhouette-suggested k: 5


In [10]:
KMEANS_K = _suggested_k  # override with any int to use a different k

_t0 = time.perf_counter()
kmeans_labels = KMeans(n_clusters=KMEANS_K, random_state=42, n_init="auto").fit_predict(peptideclm_embeddings)
peptide_voters["cluster_peptideclm"] = kmeans_labels
_runtime_c = time.perf_counter() - _t0 + _runtime_embed  # count embedding time too, not just the KMeans call

report_clustering_diagnostics(peptide_voters, "cluster_peptideclm", _runtime_c)


[cluster_peptideclm] runtime: 251.24s
[cluster_peptideclm] coverage: 12371/12371 peptides scored (100.0%)
[cluster_peptideclm] cluster count: 5
[cluster_peptideclm] cluster size distribution: min=977, median=2310, max=3952, mean=2474.2
[cluster_peptideclm] largest cluster is 31.9% of scored peptides


## Save the embeddings (for reuse in notebook 05's UMAP visualization) and the voter column

In [11]:
np.save(CLUSTERING_DIR / "peptideclm_embeddings.npy", peptideclm_embeddings)
save_peptide_voters(peptide_voters)


saved /Users/lukajin/PycharmProjects/soamp/data/clustering/peptide_voters.parquet (12371 rows, columns: ['peptide_id', 'sequence', 'smiles', 'has_noncanonical', 'bond_type', 'is_linear', 'cluster_fingerprint', 'cluster_descriptor', 'qmap_input_sequence', 'qmap_scoreable', 'cluster_qmap', 'cluster_peptideclm'])
